# 04. v10 엣지 피처 (팀원 SALT-GNN 전처리 재현)

팀원 노트북 `01_원본전처리` 의 거래 피처(`raw_common.build_transaction_features` + `preprocessing.build_edges`)를
**우리 `01_prepare_parquet` 출력**(`parquets/<DATASET>/transactions.parquet`, `kyc.parquet`)에서 같은 SQL로 다시 만든다.
목적: GNN 과 **같은 입력**으로 LightGBM 을 돌려, 무패턴을 못 찾는 원인이 피처인지 모델인지 가르기.

| 순서 | 노트북 | 출력 (`parquets/<DATASET>/nonpaten_vs_normal/v10_lgbm/`) |
|---|---|---|
| **04** | 엣지 피처 17 + 결제유형 코드 + port | `edge_features.parquet` (edge_id 순) |
| 05 | 계좌 피처 19, 날짜별 마감 스냅샷 | `nodes/<날짜>.npy`, `first_days/` |
| 06 | LightGBM 입력 (구간별, 엣지 + src·dst 노드) | `lgbm/{train,val,test}.parquet` |

**분할은 우리 binary_classification 실험 기간** — Train 08-08~10-02 / Val 10-03~10-16 / Test 10-17~10-30

**팀원 코드와 똑같이 둔 것**
- 이력은 `2022-08-01` 부터 누적. 모든 피처가 그 거래 이전 거래만 쓰므로 팀원 값과 같다 (10/31·11월은 쓰지 않음)
- 시간 창은 **같은 분 제외** (`RANGE ... AND INTERVAL 1 MINUTE PRECEDING`). `pair_history` · `reverse` · `pair_age` 는 **어제까지**(ASOF `day >`)
- 금액 z 두 개는 **8/1~8/7 거래로만** 맞춘 통화별 보정값 (v10 패키지 보정표 아님 — `02_build_features` 와 다른 점)
- country 는 은행명 `"<국가> Bank #"` 에서, 이름에 crypto 가 들어간 은행은 빈 값. entity_type 은 `"<유형> #번호"` 에서

| # | 피처 | 정의 | GNN 사용 |
|---|---|---|---|
| 1 | pair_prior_count | ln(1 + 이전 A→B 건수) | ○ |
| 2 | pair_gap_minutes | 직전 A→B 이후 분, ln(2+분), 없으면 0 | ○ |
| 3 | pair_count_1h | ln(1 + 직전 60분 A→B 건수) | ○ |
| 4–5 | sender_gap_minutes / sender_count_1h | 송신 계좌의 직전 송금 간격 / 60분 송금 수 | ○ |
| 6–7 | receiver_gap_minutes / receiver_count_1h | 수신 계좌의 직전 입금 간격 / 60분 입금 수 | ○ |
| 8 | paid_currency_z | 통화별 강건 z (scale·중앙값·IQR), ±8 | ○ |
| 9 | pair_history_count | ln(1 + 어제까지 A→B 누적) | × |
| 10 | reverse_history_count | ln(1 + 어제까지 B→A 누적) | ○ |
| 11 | pair_age_at_day_start | 어제까지 마지막 A→B → 오늘 0시까지 분, ln(2+분) | ○ |
| 12 | same_entity | 두 계좌 소유주 같음 | × |
| 13–14 | same_known_country / both_countries_known | 국가 같음 / 둘 다 알려짐 | ○ |
| 15–16 | same_currency / is_self_transfer | 지불=수취 통화 / A=B | ○ |
| 17 | amt_z_pay_ccy | 통화별 ln(1+금액) 평균·표준편차 z, ±10 | × |
| + | payment_format_code, in_port, out_port | 결제유형 0~7 (GNN 은 one-hot 8열), 쌍의 첫 거래 순서 | ○ |

번호 체계(edge_id·node_id)는 우리 `01_prepare_parquet` 것이라 port 는 **같은 분에 처음 생긴 쌍끼리의 순서**만 팀원과 다를 수 있다.
합성 데이터(2.3만 건, 같은 행 순서)에서는 17개·port·노드 19개가 팀원 코드 출력과 전부 일치(최대 차 2.4e-7, float32 합산 순서)함을 확인했다.

**시간·공간 (HI-Large)**: `02_build_features` 기록 기준 창 계산 약 11분 + 조립 약 18분. 작업 DuckDB 와 스필에 수십 GB 디스크.

In [ ]:
# 패키지가 없을 때만
# %pip install --break-system-packages -q duckdb pyarrow pandas numpy

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))                    # code/nonpaten_vs_normal 에서 연다
import importlib, dv_lib as D; importlib.reload(D)

DATASET = "HI-Large"
THREADS, MEMORY = 8, "10GB"        # DuckDB. 컨테이너 한도(약 25GB) 안에서 파이썬 배열과 나눠 쓰도록 절반 이하
P = D.Paths(DATASET)
print(P)

STRICT = True                      # 구간별 거래 수가 binary_classification 실험과 다르면 멈춤
work = P.out / "work_edge.duckdb"
for stale in (work, Path(str(work) + ".wal")):
    stale.unlink(missing_ok=True)
con = D.connect(P, THREADS, MEMORY, db_file=work)

## 1. 입력 — 관측 구간 거래 `obs`, 계좌표 `kyc_t`

`obs` 에는 **라벨 열이 없다** (피처 계산이 라벨을 볼 수 없게). 중복 대표만, 관측 구간만.
구간별 거래 수가 binary_classification 04 노트북에 찍힌 수(train 102,336,629 / val 24,413,046 / test 26,808,230)와 같으면 같은 모집단이다.

In [ ]:
D.register_inputs(con, P)
counts = D.split_counts(con)
display(counts)
print("관측 거래", f"{con.execute('SELECT count(*) FROM obs').fetchone()[0]:,}",
      "| 계좌", f"{con.execute('SELECT count(*) FROM kyc_t').fetchone()[0]:,}")
display(con.execute("SELECT entity_type, count(*) n FROM kyc_t GROUP BY 1 ORDER BY n DESC").df())
display(con.execute("SELECT country = '' AS unknown_country, count(*) n FROM kyc_t GROUP BY 1").df())
if STRICT:
    assert counts.same.all(), "구간 거래 수가 binary_classification 실험과 다르다 — 01_prepare_parquet 출력 확인"

## 2. 계산 — 단계별 SQL

`D.build_edge_steps` 의 각 단계를 순서대로 실행한다 (SQL 은 `dv_lib.py` 에 그대로 있다).
결과 `edge_features.parquet` 이 이미 있으면 건너뛴다 — 다시 만들려면 파일을 지우고 실행.

In [ ]:
if P.edge.exists():
    print("이미 있음 → 건너뜀:", P.edge)
else:
    t_all = time.time()
    for name, sql in D.build_edge_steps(con):
        t0 = time.time(); D.log(name); con.execute(sql); D.log(f"  {time.time()-t0:,.0f}s")
    t0 = time.time(); D.export_edges(con, P); D.log(f"parquet 저장 {time.time()-t0:,.0f}s (전체 {time.time()-t_all:,.0f}s)")
display(con.execute("SELECT * FROM robust ORDER BY currency").df() if "robust" in con.execute("SHOW TABLES").df().name.values
        else "robust 표 없음 (건너뜀)")

## 3. 확인

1. 행 수 = 관측 거래 수, 비유한값 0
2. 라벨별(정상 / 패턴 / 무패턴) 평균·0 비율 — 무패턴 비교 분석에서 본 방향이 나오는지
3. 송신 계좌 하나의 타임라인으로 창이 의도대로 움직이는지

In [ ]:
bad = " OR ".join(f'"{n}" IS NULL OR NOT isfinite("{n}")' for n in D.EDGE_FEATURES)
n_edge, n_bad = con.execute(f"SELECT count(*), count(*) FILTER (WHERE {bad}) FROM read_parquet({D.q(P.edge)})").fetchone()
n_obs = con.execute("SELECT count(*) FROM obs").fetchone()[0]
print(f"edge_features {n_edge:,} 행 (obs {n_obs:,}), 비유한값 {n_bad}")
assert n_edge == n_obs and n_bad == 0

In [ ]:
# 정상(0) / 패턴(1~8) / 무패턴(9) 별 평균 — 라벨은 여기서 처음 붙인다
D.register_labels(con, P)
grp = "CASE WHEN l.cls = 0 THEN 'normal' WHEN l.cls = 9 THEN 'nonpattern' ELSE 'pattern' END"
mean = con.execute(f"""SELECT {grp} AS g, count(*) n, {", ".join(f'avg(e."{c}") AS "{c}"' for c in D.EDGE_FEATURES)}
                      FROM read_parquet({D.q(P.edge)}) e JOIN labels l USING (edge_id) GROUP BY 1""").df().set_index("g").T
zero = con.execute(f"""SELECT {grp} AS g, {", ".join(f'avg((e."{c}" = 0)::INT) AS "{c}"' for c in D.EDGE_FEATURES)}
                      FROM read_parquet({D.q(P.edge)}) e JOIN labels l USING (edge_id) GROUP BY 1""").df().set_index("g").T
pd.concat({"mean": mean, "zero_rate": zero}, axis=1).round(3)

In [ ]:
# 하루 6~12건 보내는 송신 계좌 하나 — pair/sender 값이 과거(같은 분 제외)로만 움직이는지
acct, d = con.execute(f"""SELECT src, day FROM read_parquet({D.q(P.edge)}) WHERE day >= DATE '2022-09-01'
                          GROUP BY src, day HAVING count(*) BETWEEN 6 AND 12 ORDER BY day, src LIMIT 1""").fetchone()
tl = con.execute(f"""SELECT e.edge_id, o.ts, e.dst, e.pair_prior_count, e.pair_count_1h, e.sender_gap_minutes, e.sender_count_1h,
                            e.pair_history_count, e.pair_age_at_day_start
                     FROM read_parquet({D.q(P.edge)}) e JOIN obs o USING (edge_id)
                     WHERE e.src = {acct} AND e.day BETWEEN DATE '{d}' - INTERVAL 1 DAY AND DATE '{d}' + INTERVAL 1 DAY
                     ORDER BY o.ts, e.edge_id""").df()
tl["pair_prior_n"] = np.expm1(tl.pair_prior_count).round().astype(int)
tl["sender_gap(분)"] = np.where(tl.sender_gap_minutes > 0, np.exp(tl.sender_gap_minutes) - 2, np.nan).round()
tl

### (선택) `02_build_features` 의 `features_test` 와 교차 확인

정의가 같은 피처는 값이 같아야 한다 (`paid_currency_z`·`amt_z_pay_ccy` 는 보정표가, country 두 개는 crypto 규칙이 달라 제외).
`features_test/edge_features.parquet` 이 없으면 건너뛴다. 하루치(10/20)만 비교.

In [ ]:
other = P.base / "features_test" / "edge_features.parquet"
SAME = ["pair_prior_count", "pair_gap_minutes", "pair_count_1h", "sender_gap_minutes", "sender_count_1h", "receiver_gap_minutes",
        "receiver_count_1h", "pair_history_count", "reverse_history_count", "pair_age_at_day_start", "same_entity", "same_currency", "is_self_transfer"]
if other.exists():
    cmp = con.execute(f"""SELECT count(*) n, {", ".join(f'max(abs(a."{c}" - b."{c}")) AS "{c}"' for c in SAME)}
                          FROM read_parquet({D.q(P.edge)}) a JOIN read_parquet({D.q(other)}) b USING (edge_id)
                          WHERE a.day = DATE '2022-10-20'""").df().T
    display(cmp.rename(columns={0: "max|diff|"}))
else:
    print("features_test 없음 → 건너뜀")

In [ ]:
D.write_meta(P, "04_edge", dict(rows=n_edge, split_counts=counts.to_dict("records"), duckdb=__import__("duckdb").__version__,
                                created=time.strftime("%Y-%m-%d %H:%M:%S")))
con.close()
for stale in (work, Path(str(work) + ".wal")):
    stale.unlink(missing_ok=True)
print(f"{P.edge}  {P.edge.stat().st_size/1e9:.2f} GB")